# Elliptic Curves on the Projective Sphere

Interactive 3D visualization of how the affine plane wraps onto a sphere
via **inverse stereographic projection**, and how different elliptic curves
look in projective space.

Three curves are compared:
- **Red**: $y^2 = x^3 + 7$ (secp256k1 / Bitcoin) — small loop at the north pole
- **Cyan**: $y^2 = x^3 - x$ (two components) — egg + branch, two separate circles
- **White**: $y^2 = x^3 - x + \frac{1}{2}$ (one component) — single dramatic loop wrapping the sphere

Drag to rotate. Hit **Spin** to auto-rotate.

---

### Visualizing the Projective Plane

The text demo above shows the *idea* — now let's **see** it.

We take the Bitcoin curve $y^2 = x^3 + 7$ in the ordinary (affine) plane and lift it
onto a unit sphere using **inverse stereographic projection** from the north pole:

$$
(u, v) \;\longmapsto\;
\left(
  \frac{2u}{u^2+v^2+1},\;
  \frac{2v}{u^2+v^2+1},\;
  \frac{u^2+v^2-1}{u^2+v^2+1}
\right)
$$

In the affine plane the two branches of the curve diverge to $\pm\infty$.
On the sphere **they close up and meet at the north pole** — that single point
*is* the point at infinity $\mathcal{O}$ that makes the curve a group.

*(Inspired by [Trustica's video](https://www.youtube.com/watch?v=Hk0Fr-k7wmQ)
on the Weierstrass curve in the projective plane.)*

In [ ]:
import numpy as np
import plotly.graph_objects as go

def inv_stereo(u, v):
    """Inverse stereographic projection: plane (u,v) → unit sphere (X,Y,Z).
    North pole (0,0,1) is the projection centre; (0,0) maps to south pole."""
    d = u**2 + v**2 + 1
    return 2*u/d, 2*v/d, (u**2 + v**2 - 1)/d

# ── grid ────────────────────────────────────────────────────────────────
span = 2 * np.pi
n_grid = 30                               # grid density (lines, not fill)
n_pts  = 200                              # points per grid line (smoothness)
vals = np.linspace(-span, span, n_grid)   # where the grid lines sit
line_t = np.linspace(-span, span, n_pts)  # parametric samples along each line

# Also build a dense fill mesh for surface colouring
n_fill = 80
fill_v = np.linspace(-span, span, n_fill)
Xf, Yf = np.meshgrid(fill_v, fill_v)
Xs_f, Ys_f, Zs_f = inv_stereo(Xf, Yf)

# Pre-build the grid lines as flat arrays with None breaks (one Scatter3d each)
def build_grid_lines(map_fn):
    """Return (gx, gy, gz) for all grid lines, with None separators."""
    gx, gy, gz = [], [], []
    for v in vals:                         # horizontal lines (constant y = v)
        xs, ys, zs = map_fn(line_t, np.full_like(line_t, v))
        gx += list(xs) + [None]
        gy += list(ys) + [None]
        gz += list(zs) + [None]
    for v in vals:                         # vertical lines   (constant x = v)
        xs, ys, zs = map_fn(np.full_like(line_t, v), line_t)
        gx += list(xs) + [None]
        gy += list(ys) + [None]
        gz += list(zs) + [None]
    return gx, gy, gz

def flat_map(u, v):
    return u, v, np.zeros_like(u)

grid_flat = build_grid_lines(flat_map)
grid_sphere = build_grid_lines(inv_stereo)

# ── Bitcoin curve  y² = x³ + 7  (single closed loop) ───────────────────
x_min = -(7 ** (1/3))
t_near = np.linspace(x_min + 1e-8, 5, 1500)
t_far  = np.geomspace(5, 800, 1500)       # geometric spacing → denser near 5
t_all = np.concatenate([t_near, t_far])
t_all = t_all[t_all**3 + 7 >= 0]
y_all = np.sqrt(t_all**3 + 7)

# Trace: upper branch forward, then lower branch in reverse → closed loop
loop_u = np.concatenate([t_all, t_all[::-1]])
loop_v = np.concatenate([y_all, -y_all[::-1]])

# On the sphere (lifted slightly so it sits above the surface)
_lx, _ly, _lz = inv_stereo(loop_u, loop_v)
R_LIFT = 1.012
curve_sx = R_LIFT * np.array(_lx)
curve_sy = R_LIFT * np.array(_ly)
curve_sz = R_LIFT * np.array(_lz)

# Clipped for the flat-plane view
mask = (t_all <= span) & (y_all <= span)
t_f = t_all[mask]
yp_f, yn_f = y_all[mask], -y_all[mask]
flat_loop_u = np.concatenate([t_f, t_f[::-1]])
flat_loop_v = np.concatenate([yp_f, yn_f[::-1]])

GRID_COLOR = "rgba(0,0,0,0.45)"
GRID_W = 1
CURVE_W = 5

# ── y² = x³ − x  (two components — egg + branch) ───────────────────────
t_egg = np.linspace(-1 + 1e-8, -1e-8, 800)
y_egg = np.sqrt(t_egg**3 - t_egg)
egg_u = np.concatenate([t_egg, t_egg[::-1]])
egg_v = np.concatenate([y_egg, -y_egg[::-1]])
_ex, _ey, _ez = inv_stereo(egg_u, egg_v)

t_br_near = np.linspace(1 + 1e-8, 5, 1000)
t_br_far  = np.geomspace(5, 600, 1200)
t_br = np.concatenate([t_br_near, t_br_far])
y_br = np.sqrt(t_br**3 - t_br)
br_u = np.concatenate([t_br, t_br[::-1]])
br_v = np.concatenate([y_br, -y_br[::-1]])
_bx, _by, _bz = inv_stereo(br_u, br_v)

R_LIFT = 1.012
egg_sx, egg_sy, egg_sz = R_LIFT*np.array(_ex), R_LIFT*np.array(_ey), R_LIFT*np.array(_ez)
br_sx, br_sy, br_sz    = R_LIFT*np.array(_bx), R_LIFT*np.array(_by), R_LIFT*np.array(_bz)

# ── y² = x³ − x + 0.5  (ONE component — single big loop around sphere) ─
a_w, b_w = -1, 0.5
roots_w = np.roots([1, 0, a_w, b_w])
x_min_w = roots_w[np.abs(roots_w.imag) < 1e-10].real.min()
t_w_near = np.linspace(x_min_w + 1e-8, 5, 2000)
t_w_far  = np.geomspace(5, 800, 1500)
t_w = np.concatenate([t_w_near, t_w_far])
rhs_w = t_w**3 + a_w*t_w + b_w
t_w = t_w[rhs_w >= 0]
y_w = np.sqrt(t_w**3 + a_w*t_w + b_w)
wrap_u = np.concatenate([t_w, t_w[::-1]])
wrap_v = np.concatenate([y_w, -y_w[::-1]])
_wx, _wy, _wz = inv_stereo(wrap_u, wrap_v)
wrap_sx = R_LIFT * np.array(_wx)
wrap_sy = R_LIFT * np.array(_wy)
wrap_sz = R_LIFT * np.array(_wz)

# ── y² = x³ + 7  (secp256k1, single component) ────────────────────────
x_min_btc = -(7 ** (1/3))
t_btc_near = np.linspace(x_min_btc + 1e-8, 5, 1500)
t_btc_far  = np.geomspace(5, 800, 1500)
t_btc = np.concatenate([t_btc_near, t_btc_far])
t_btc = t_btc[t_btc**3 + 7 >= 0]
y_btc = np.sqrt(t_btc**3 + 7)
btc_u = np.concatenate([t_btc, t_btc[::-1]])
btc_v = np.concatenate([y_btc, -y_btc[::-1]])
_cx, _cy, _cz = inv_stereo(btc_u, btc_v)
btc_sx, btc_sy, btc_sz = R_LIFT*np.array(_cx), R_LIFT*np.array(_cy), R_LIFT*np.array(_cz)

# flat-plane clipped versions
mask_btc = (t_btc <= span) & (y_btc <= span)
t_bf, yp_bf = t_btc[mask_btc], y_btc[mask_btc]
flat_btc_u = np.concatenate([t_bf, t_bf[::-1]])
flat_btc_v = np.concatenate([yp_bf, -yp_bf[::-1]])

print("Shared data ready — run the next two cells.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  AFFINE PLANE — flat grid  +  y² = x³ + 7
# ═══════════════════════════════════════════════════════════════════════
fig_a = go.Figure()

# coloured fill (no contours — we draw the real grid lines ourselves)
fig_a.add_trace(go.Surface(
    x=Xf, y=Yf, z=np.zeros_like(Xf),
    surfacecolor=Zs_f,
    colorscale="Viridis", showscale=False, opacity=0.80,
))

# parametric grid lines
gx, gy, gz = grid_flat
fig_a.add_trace(go.Scatter3d(
    x=gx, y=gy, z=gz,
    mode="lines", line=dict(color=GRID_COLOR, width=GRID_W),
    showlegend=False, hoverinfo="skip",
))

# secp256k1:  y² = x³ + 7  (red)
fig_a.add_trace(go.Scatter3d(
    x=flat_btc_u, y=flat_btc_v, z=np.full(len(flat_btc_u), 0.12),
    mode="lines", line=dict(color="rgb(255,34,0)", width=CURVE_W),
    name="y² = x³ + 7", showlegend=True,
))

# y² = x³ − x  egg (cyan) — clipped to visible range
fig_a.add_trace(go.Scatter3d(
    x=egg_u, y=egg_v, z=np.full(len(egg_u), 0.12),
    mode="lines", line=dict(color="rgb(0,200,255)", width=CURVE_W),
    name="y² = x³ − x (egg)", showlegend=True,
))
# y² = x³ − x  branch (cyan) — clipped
mask_br_flat = (t_br <= span) & (y_br <= span)
t_brf, y_brf = t_br[mask_br_flat], y_br[mask_br_flat]
fig_a.add_trace(go.Scatter3d(
    x=np.concatenate([t_brf, t_brf[::-1]]),
    y=np.concatenate([y_brf, -y_brf[::-1]]),
    z=np.full(2*len(t_brf), 0.12),
    mode="lines", line=dict(color="rgb(0,200,255)", width=CURVE_W),
    name="y² = x³ − x (branch)", showlegend=True,
))

fig_a.update_layout(
    title="Affine plane  —  two curves on the grid",
    scene=dict(
        xaxis_title="x", yaxis_title="y", zaxis_title="",
        camera=dict(eye=dict(x=0.4, y=-1.1, z=1.7)),
        zaxis=dict(range=[-0.5, 0.5], showticklabels=False),
        aspectmode="manual", aspectratio=dict(x=1.2, y=1.2, z=0.10),
    ),
    height=650, width=950,
    margin=dict(l=0, r=0, t=40, b=0),
)
fig_a.show()
print("Uniform grid squares. The colour shows where each square lands on the sphere (next cell).")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  PROJECTIVE SPHERE — same grid warped via stereographic projection
#  Drag to rotate, or hit ▶ Spin.
# ═══════════════════════════════════════════════════════════════════════
fig_s = go.Figure()

# coloured fill (no built-in contours)
fig_s.add_trace(go.Surface(
    x=Xs_f, y=Ys_f, z=Zs_f,
    surfacecolor=Zs_f,
    colorscale="Viridis", showscale=False, opacity=0.65,
))

# real warped grid lines (each straight line on the plane becomes a curve here)
gx, gy, gz = grid_sphere
fig_s.add_trace(go.Scatter3d(
    x=gx, y=gy, z=gz,
    mode="lines", line=dict(color=GRID_COLOR, width=GRID_W),
    showlegend=False, hoverinfo="skip",
))

# secp256k1: y² = x³ + 7  (red — hugs upper hemisphere)
fig_s.add_trace(go.Scatter3d(
    x=btc_sx, y=btc_sy, z=btc_sz,
    mode="lines", line=dict(color="rgb(255,34,0)", width=CURVE_W),
    name="y² = x³ + 7  (secp256k1)", showlegend=True,
))

# y² = x³ − x  egg (cyan — loops through lower hemisphere)
fig_s.add_trace(go.Scatter3d(
    x=egg_sx, y=egg_sy, z=egg_sz,
    mode="lines", line=dict(color="rgb(0,200,255)", width=CURVE_W),
    name="y² = x³ − x  (egg)", showlegend=True,
))

# y² = x³ − x  branch (cyan — sweeps up to north pole)
fig_s.add_trace(go.Scatter3d(
    x=br_sx, y=br_sy, z=br_sz,
    mode="lines", line=dict(color="rgb(0,200,255)", width=CURVE_W),
    name="y² = x³ − x  (branch)", showlegend=True,
))

# y² = x³ − x + 0.5  (white — single big loop wrapping the whole sphere)
fig_s.add_trace(go.Scatter3d(
    x=wrap_sx, y=wrap_sy, z=wrap_sz,
    mode="lines", line=dict(color="white", width=CURVE_W + 1),
    name="y² = x³ − x + ½  (single loop)", showlegend=True,
))

# point at infinity
fig_s.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[1.03],
    mode="markers+text",
    marker=dict(size=8, color="white", line=dict(color="red", width=3)),
    text=["  𝒪 (point at ∞)"], textposition="top right",
    textfont=dict(size=14, color="red", family="serif"),
    showlegend=False,
))

fig_s.update_layout(
    title="Projective sphere  —  drag to rotate  /  ▶ Spin",
    scene=dict(
        xaxis_title="X", yaxis_title="Y", zaxis_title="Z",
        camera=dict(eye=dict(x=1.4, y=-1.4, z=0.6)),
        aspectmode="data",
    ),
    height=700, width=950,
    margin=dict(l=0, r=0, t=40, b=0),
)

# auto-spin
n_frames = 72
frames = []
for i in range(n_frames):
    a = 2 * np.pi * i / n_frames
    frames.append(go.Frame(
        layout=dict(scene=dict(camera=dict(
            eye=dict(x=1.8*np.cos(a), y=1.8*np.sin(a), z=0.6)))),
        name=str(i),
    ))
fig_s.frames = frames
fig_s.update_layout(updatemenus=[dict(
    type="buttons", showactive=False,
    x=0.95, y=0.02, xanchor="right", yanchor="bottom",
    buttons=[
        dict(label="▶ Spin", method="animate",
             args=[None, dict(frame=dict(duration=80, redraw=True),
                              fromcurrent=True, mode="immediate",
                              transition=dict(duration=0))]),
        dict(label="⏸ Stop", method="animate",
             args=[[None], dict(frame=dict(duration=0, redraw=False),
                                mode="immediate")]),
    ],
)])

fig_s.show()
print("Red   = y² = x³ + 7      (secp256k1)  — small loop hugging the north pole")
print("Cyan  = y² = x³ − x      (2 components — egg + branch, two separate circles)")
print("White = y² = x³ − x + ½  (1 component — single loop sweeping around the sphere)")
print("\nToggle each in the legend. The white curve is what the Trustica video shows.")